# PerGAZE GazeformerISP-S on Google Colab: one GPU

Select **Runtime > Change runtime type > GPU**, then run cells in order.
Upload the PerGAZE dataset, base Gazeformer checkpoint and original text NPZ to
Google Drive. Edit their paths below. This notebook always uses only cuda:0.

Defaults: K=5, seed=10, 3 adaptation epochs. Only new subject embedding rows
are updated. TA/TP with the same numeric ID are different people: nine unseen
identities select 45 support samples at K=5. The frozen COCO ResNet50 processes
images online during training/evaluation; no image feature cache is produced.

Every epoch evaluates test_unseen and saves the best checkpoint by the harmonic
mean of the two ScanMatch scores. As requested, test_unseen selects the epoch;
the best report is not an untouched held-out test estimate.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import subprocess
import sys
from pathlib import Path

CODE = Path("/content/IndividualScanpath")
if (CODE / ".git").is_dir():
    subprocess.run(["git", "-C", str(CODE), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/ntnghia06/IndividualScanpath.git", str(CODE)], check=True)
SRC = CODE / "PerGAZE/GazeformerISP-S/src"
assert (SRC / "train.py").is_file()


In [ ]:
%pip install -q "numpy>=1.24" "Pillow>=9.1" "scipy>=1.10" "tqdm>=4.65" multimatch-gaze sentence-transformers


In [ ]:
import torch
import torchvision

assert torch.cuda.is_available(), "Select GPU in Runtime > Change runtime type"
GPU_IDS = [0]  # Use exactly one GPU, even if additional GPUs are visible.
print("PyTorch:", torch.__version__, "torchvision:", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory GiB:", torch.cuda.get_device_properties(0).total_memory / 2**30)


## Configure paths and adaptation

Example Google Drive layout:

```text
MyDrive/PerGAZE/
  dataset/
    support.json
    test_unseen.json
    images/
  pergaze_gazeformer_run/checkpoints/best.pth
  gazeformer_task_embeddings.npz
```

Edit DATA_SOURCE, CHECKPOINT_SOURCE and TEXT_SOURCE. Gazeformer does not require
attention_reasoning maps. COPY_DATA_TO_LOCAL=True copies original images and
JSON files to /content to reduce small-file Drive reads. This copies input
images, not extracted features. Set False to read images directly from Drive.

RUN is on Drive so last/best checkpoints and report persist after each epoch.
After a session interruption, rerun setup with RESUME=True and identical K,
seed, batch and input files. An interrupted epoch restarts from the last saved
completed epoch. Use a different RUN to start another experiment.


In [ ]:
K = 5
SEED = 10
EPOCHS = 3
TRAIN_BATCH = 2
TEST_BATCH = 4
WORKERS = 2
RESUME = False
COPY_DATA_TO_LOCAL = True

DRIVE_ROOT = Path("/content/drive/MyDrive/PerGAZE")
DATA_SOURCE = DRIVE_ROOT / "dataset"
CHECKPOINT_SOURCE = DRIVE_ROOT / "pergaze_gazeformer_run/checkpoints/best.pth"
TEXT_SOURCE = DRIVE_ROOT / "gazeformer_task_embeddings.npz"
RUN = DRIVE_ROOT / "runs" / f"gazeformer_s_k{K}_seed{SEED}_colab_1gpu"
LOCAL_ROOT = Path("/content/pergaze_gazeformer_s_inputs")

for name in ("support.json", "test_unseen.json", "images"):
    assert (DATA_SOURCE / name).exists(), f"Missing: {DATA_SOURCE / name}"
assert CHECKPOINT_SOURCE.is_file(), f"Missing base checkpoint: {CHECKPOINT_SOURCE}"
assert TEXT_SOURCE.is_file(), f"Missing original text NPZ: {TEXT_SOURCE}"
assert min(K, EPOCHS, TRAIN_BATCH, TEST_BATCH) >= 1 and WORKERS >= 0
if RESUME:
    assert (RUN / "checkpoints/checkpoint.pth").is_file(), f"No resume checkpoint at {RUN}"


In [ ]:
import shutil

LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
if COPY_DATA_TO_LOCAL:
    DATA = LOCAL_ROOT / "dataset"
    DATA.mkdir(parents=True, exist_ok=True)
    for name in ("support.json", "test_unseen.json"):
        shutil.copy2(DATA_SOURCE / name, DATA / name)
    print("Copying original images from Drive to /content; this can take a few minutes...")
    shutil.copytree(DATA_SOURCE / "images", DATA / "images", dirs_exist_ok=True)
else:
    DATA = DATA_SOURCE

# Stage large inputs once to speed up checkpoint loading and hashing.
CHECKPOINT = LOCAL_ROOT / "base_checkpoint.pth"
TEXT_EMBEDDINGS = LOCAL_ROOT / "gazeformer_task_embeddings.npz"
shutil.copy2(CHECKPOINT_SOURCE, CHECKPOINT)
shutil.copy2(TEXT_SOURCE, TEXT_EMBEDDINGS)
RUN.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA)
print("Checkpoint:", CHECKPOINT)
print("Text embedding:", TEXT_EMBEDDINGS)
print("Persistent Drive output:", RUN)


In [ ]:
COMMON = [
    "--checkpoint", str(CHECKPOINT),
    "--support_file", str(DATA / "support.json"),
    "--test_file", str(DATA / "test_unseen.json"),
    "--img_dir", str(DATA / "images"),
    "--text_embeddings", str(TEXT_EMBEDDINGS),
    "--k", str(K), "--seed", str(SEED), "--epoch", str(EPOCHS),
    "--device", "cuda:0", "--gpu_ids", "0",
]

def run_script(script, *arguments):
    subprocess.run([sys.executable, "-u", str(SRC / script), *COMMON, *arguments], check=True)


## Finetune and evaluate

Evaluate the baseline, finetune K samples per identity, and evaluate all unseen
records after each epoch. Train, eval and retrieval show progress bars.
Only GPU 0 is used. Online checkpoints contain backbone weights; compatible
head-only checkpoints download the frozen pretrained COCO backbone if needed.
Internet is needed for cloning, dependency installation and missing pretrained
weights. The text embedding archive is reused, not regenerated during finetuning.


In [ ]:
arguments = ["--log_root", str(RUN), "--workers", str(WORKERS),
             "--batch", str(TRAIN_BATCH), "--test_batch", str(TEST_BATCH)]
if RESUME:
    arguments.append("--resume")
run_script("train.py", *arguments)


In [ ]:
import json
report = json.loads((RUN / "report.json").read_text(encoding="utf-8"))
print("Support samples:", report["support_samples"])
print("Best epoch:", report["best_epoch"], "score:", report["best_score"])
print(json.dumps(report["best_metrics"]["overall"], indent=2))
print(json.dumps(report["best_metrics"].get("retrieval", {}).get("by_condition", {}), indent=2))
print("Full report:", RUN / "report.json")


## Re-evaluate the best checkpoint (optional)

Uncomment to run. The later --checkpoint overrides the base checkpoint path.
Results are appended to the reevaluation section of the same report.json.


In [ ]:
# run_script("test.py", "--checkpoint", str(RUN / "checkpoints/best.pth"),
#            "--log_root", str(RUN), "--workers", str(WORKERS),
#            "--test_batch", str(TEST_BATCH))


## Outputs and optional download

Drive already contains checkpoints/best.pth, checkpoints/checkpoint.pth and
report.json. Keep the entire run directory for resume, plus the original base
checkpoint and text NPZ. The next cell prints the output path; uncomment its
remaining lines to create a ZIP in /content and download it to your computer.


In [ ]:
print("Outputs saved on Google Drive:", RUN)
# archive = shutil.make_archive(str(LOCAL_ROOT / f"gazeformer_s_k{K}_seed{SEED}"),
#                              "zip", root_dir=RUN.parent, base_dir=RUN.name)
# from google.colab import files
# files.download(archive)
